# PyTorch 调试与可复现性

## 学习目标

通过故障复现掌握 shape、dtype、device、梯度、train/eval 和随机种子问题的定位流程。

## 概念模型

先确认输入/输出契约，再确认参数注册和梯度路径，最后检查优化器、学习率和数据。每个实验都包含错误、诊断和修复。

In [ ]:
import torch
from torch import nn
from common.runtime import seed_everything, choose_device

seed_everything(42)
device = choose_device('cpu')
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2)).to(device)
x = torch.randn(8, 4, device=device)
y = torch.randint(0, 2, (8,), device=device)
print(torch.__version__, device, x.shape, x.dtype, y.dtype)

### 实验 1：dtype、shape 和参数注册

**实验目的**：主动触发高频契约错误。`CrossEntropyLoss` 的类别索引标签必须是 long、shape 为 `(batch,)`；float 标签会报 dtype 错误。

普通 Python list 不注册其中的层，参数不会进入 `parameters()`、state_dict 或设备移动；`ModuleList` 会注册。调试时同时打印 dtype、shape、device 与 `named_parameters()`。


In [ ]:
loss_fn = nn.CrossEntropyLoss()
try:
    loss_fn(model(x), y.float())
except RuntimeError as error:
    print('expected dtype error:', type(error).__name__)

bad_layers = nn.Module()
bad_layers.layers = [nn.Linear(4, 4)]
good_layers = nn.ModuleList([nn.Linear(4, 4)])
print('unregistered:', sum(p.numel() for p in bad_layers.parameters()), 'registered:', sum(p.numel() for p in good_layers.parameters()))
assert sum(p.numel() for p in good_layers.parameters()) > 0

### 实验 2：梯度与小 batch 过拟合

**实验目的**：让模型反复拟合同一小批数据，确认 loss 有限、梯度存在、参数更新链路连通。若简单模型无法降低 loss，优先检查标签、loss、参数注册、梯度开关和 optimizer。

`clip_grad_norm_` 返回裁剪前总范数。裁剪可限制更新，但可能掩盖根因；小 batch 过拟合通过也不代表真实数据能泛化。


In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)
losses = []
for _ in range(30):
    optimizer.zero_grad(set_to_none=True)
    loss = loss_fn(model(x), y)
    assert torch.isfinite(loss)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    losses.append(loss.item())
print('loss:', losses[0], '->', losses[-1], 'grad_norm:', float(grad_norm))
assert losses[-1] < losses[0]

### 实验 3：train/eval、计时和复现

**实验目的**：验证 Dropout 模式差异、重复设种子可重现随机张量，以及设备计时的同步边界。`eval()` 不关闭梯度，推理仍需 inference mode。

相同 seed 只在相同环境和随机调用顺序下提供复现基础；多 worker、非确定算子和版本变化仍会造成差异。CUDA 计时边界必须同步。


In [ ]:
drop = nn.Dropout(p=0.5)
drop.train(); train_values = drop(torch.ones(1000))
drop.eval(); eval_values = drop(torch.ones(1000))
assert not torch.equal(train_values, eval_values)
seed_everything(7); first = torch.randn(4)
seed_everything(7); second = torch.randn(4)
torch.testing.assert_close(first, second)
print('train/eval and reproducibility checks passed')

## 官方教程补充

**对应官方源文件：** `recipes_source/debug_mode_tutorial.py`、`recipes_source/torch_logs.py`、`intermediate_source/visualizing_gradients_tutorial.py`

官方调试建议先缩小问题：固定 seed、用单 batch 过拟合、断言 shape/dtype/device 和有限值，再检查梯度与参数是否更新。可复现不等于跨平台逐位一致，确定性算法可能降低性能且仍受版本/硬件影响。遇到编译或分布式问题时使用 PyTorch 日志接口增加可观测性，而不是只依赖最终异常。

**验证练习：** 找到上面源文件中的对应 API，先写出输入、输出和状态变化，再运行本 notebook 的相关实验；如果行为不同，优先检查本地 PyTorch 版本、设备能力和输入契约。

<!-- official-pytorch-supplement-v1 -->

## 检查点

说明为什么 `grad is None`、标签 dtype 错误和 device mismatch 属于不同类别的问题，并写出各自的第一条检查命令。

## 试一试

故意把学习率改成过大值、把标签打乱、把验证阶段改成 `train()`，分别记录 loss、指标或输出的变化。

## 常见错误与调试

固定顺序：打印 shape/dtype/device -> 检查任务与损失匹配 -> 小 batch 过拟合 -> 检查参数和梯度 -> 检查学习率与数据。